In [0]:
print("Hello World")
df= spark.read.option("header","true").option("inferSchema","true").csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/raw/sales/sales.csv")
display()

Hello World


In [0]:
df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- UnitPrice: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- GrossSales: integer (nullable = true)
 |-- DiscountAmount: double (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
display(df.limit(10))

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
O00001,2024-09-22,C0080,P002,North,5.0,65000,0.1,325000,32500.0,292500.0,220527.82,71972.18
O00002,2025-08-27,C0045,P015,West,1.0,500,0.0,500,0.0,500.0,396.17,103.83
O00003,2024-01-08,C0099,P003,East,2.0,42000,0.05,84000,4200.0,79800.0,47199.84,32600.16
O00004,2025-10-17,C0007,P007,South,2.0,9000,0.0,18000,0.0,18000.0,11553.08,6446.92
O00005,2025-01-31,C0018,P004,South,1.0,22000,0.0,22000,0.0,22000.0,14187.01,7812.99
O00006,2024-04-12,C0097,P013,East,1.0,10500,0.1,10500,1050.0,9450.0,6739.26,2710.74
O00007,2024-08-08,C0029,P006,West,1.0,3200,0.1,3200,320.0,2880.0,2181.36,698.64
O00008,2025-02-09,C0032,P012,West,3.0,14500,0.1,43500,4350.0,39150.0,33210.33,5939.67
O00009,2024-08-18,C0080,P002,South,1.0,65000,0.0,65000,0.0,65000.0,41328.84,23671.16
O00010,2024-03-20,C0019,P007,West,1.0,9000,0.05,9000,450.0,8550.0,6759.09,1790.91


In [0]:
print("Total rows:",df.count())

Total rows: 2502


In [0]:
print("Total rows:",df.count())
print("Distinct Rows:",df.distinct().count())



Total rows: 2502
Distinct Rows: 2500


In [0]:
from pyspark.sql.functions import col,sum
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).display()

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
0,0,0,0,1,1,0,0,0,0,0,0,0


In [0]:
df_clean=df.dropDuplicates()

In [0]:
print("Original rows:",df.count())
print("Rows after deleting duplicates:",df_clean.count())

Original rows: 2502
Rows after deleting duplicates: 2500


In [0]:
from pyspark.sql.functions import coalesce,lit
df_clean=df_clean.withColumn("Region",coalesce(col("Region"), lit("Unknown")))
df_clean.filter(col("Region")=="Unknown").display()


OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
O00011,2025-05-01,C0052,P014,Unknown,1.0,700,0.02,700,14.0,686.0,412.68,273.32


In [0]:
from pyspark.sql.functions import col, coalesce, lit

df = df_clean.withColumn("quantity", coalesce(col("quantity"), lit(0.0)))
df_clean.filter(col("Quantity")=="0.0").display()


OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit


In [0]:
print("rows;",df_clean.count())
display(df_clean)


rows; 2500


OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
O00029,2024-04-13,C0063,P003,North,3.0,42000,0.1,126000,12600.0,113400.0,78421.69,34978.31
O00041,2025-04-09,C0044,P005,North,1.0,4500,0.0,4500,0.0,4500.0,3627.53,872.47
O00073,2024-10-26,C0082,P006,South,1.0,3200,0.0,3200,0.0,3200.0,2332.84,867.16
O00078,2025-06-29,C0094,P004,North,5.0,22000,0.02,110000,2200.0,107800.0,62698.39,45101.61
O00109,2025-11-14,C0013,P009,North,2.0,1800,0.02,3600,72.0,3528.0,2345.36,1182.64
O00132,2025-06-24,C0068,P004,South,5.0,22000,0.02,110000,2200.0,107800.0,85572.22,22227.78
O00135,2024-11-13,C0081,P008,North,5.0,12000,0.02,60000,1200.0,58800.0,36232.31,22567.69
O00170,2025-11-13,C0064,P005,North,5.0,4500,0.05,22500,1125.0,21375.0,14910.64,6464.36
O00175,2025-04-08,C0002,P001,North,1.0,85000,0.15,85000,12750.0,72250.0,51754.52,20495.48
O00180,2024-03-20,C0048,P009,South,5.0,1800,0.0,9000,0.0,9000.0,7109.29,1890.71


In [0]:
df_clean.write .mode("overwrite").option("header","true").csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/processed/sales_cleaned/")

In [0]:
display(spark.read.option("header","true").option("inferSchema","true").csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/processed/sales_cleaned/"))

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
O00029,2024-04-13,C0063,P003,North,3.0,42000,0.1,126000,12600.0,113400.0,78421.69,34978.31
O00041,2025-04-09,C0044,P005,North,1.0,4500,0.0,4500,0.0,4500.0,3627.53,872.47
O00073,2024-10-26,C0082,P006,South,1.0,3200,0.0,3200,0.0,3200.0,2332.84,867.16
O00078,2025-06-29,C0094,P004,North,5.0,22000,0.02,110000,2200.0,107800.0,62698.39,45101.61
O00109,2025-11-14,C0013,P009,North,2.0,1800,0.02,3600,72.0,3528.0,2345.36,1182.64
O00132,2025-06-24,C0068,P004,South,5.0,22000,0.02,110000,2200.0,107800.0,85572.22,22227.78
O00135,2024-11-13,C0081,P008,North,5.0,12000,0.02,60000,1200.0,58800.0,36232.31,22567.69
O00170,2025-11-13,C0064,P005,North,5.0,4500,0.05,22500,1125.0,21375.0,14910.64,6464.36
O00175,2025-04-08,C0002,P001,North,1.0,85000,0.15,85000,12750.0,72250.0,51754.52,20495.48
O00180,2024-03-20,C0048,P009,South,5.0,1800,0.0,9000,0.0,9000.0,7109.29,1890.71


In [0]:
clean_df = spark.read.option("header","true").option("inferSchema","true").csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/processed/sales_cleaned/")


In [0]:
clean_df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- UnitPrice: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- GrossSales: integer (nullable = true)
 |-- DiscountAmount: double (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
display(clean_df)

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit
O00029,2024-04-13,C0063,P003,North,3.0,42000,0.1,126000,12600.0,113400.0,78421.69,34978.31
O00041,2025-04-09,C0044,P005,North,1.0,4500,0.0,4500,0.0,4500.0,3627.53,872.47
O00073,2024-10-26,C0082,P006,South,1.0,3200,0.0,3200,0.0,3200.0,2332.84,867.16
O00078,2025-06-29,C0094,P004,North,5.0,22000,0.02,110000,2200.0,107800.0,62698.39,45101.61
O00109,2025-11-14,C0013,P009,North,2.0,1800,0.02,3600,72.0,3528.0,2345.36,1182.64
O00132,2025-06-24,C0068,P004,South,5.0,22000,0.02,110000,2200.0,107800.0,85572.22,22227.78
O00135,2024-11-13,C0081,P008,North,5.0,12000,0.02,60000,1200.0,58800.0,36232.31,22567.69
O00170,2025-11-13,C0064,P005,North,5.0,4500,0.05,22500,1125.0,21375.0,14910.64,6464.36
O00175,2025-04-08,C0002,P001,North,1.0,85000,0.15,85000,12750.0,72250.0,51754.52,20495.48
O00180,2024-03-20,C0048,P009,South,5.0,1800,0.0,9000,0.0,9000.0,7109.29,1890.71


In [0]:
from pyspark.sql.functions import col ,when
clean_df = clean_df.withColumn("profit_margin",when(col("SalesAmount")!=0,(col("Profit")/col("SalesAmount"))*100).otherwise(0))
display(clean_df)


OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit,profit_margin
O00029,2024-04-13,C0063,P003,North,3.0,42000,0.1,126000,12600.0,113400.0,78421.69,34978.31,30.84507054673721
O00041,2025-04-09,C0044,P005,North,1.0,4500,0.0,4500,0.0,4500.0,3627.53,872.47,19.388222222222222
O00073,2024-10-26,C0082,P006,South,1.0,3200,0.0,3200,0.0,3200.0,2332.84,867.16,27.09875
O00078,2025-06-29,C0094,P004,North,5.0,22000,0.02,110000,2200.0,107800.0,62698.39,45101.61,41.838228200371056
O00109,2025-11-14,C0013,P009,North,2.0,1800,0.02,3600,72.0,3528.0,2345.36,1182.64,33.52154195011338
O00132,2025-06-24,C0068,P004,South,5.0,22000,0.02,110000,2200.0,107800.0,85572.22,22227.78,20.619461966604824
O00135,2024-11-13,C0081,P008,North,5.0,12000,0.02,60000,1200.0,58800.0,36232.31,22567.69,38.38042517006803
O00170,2025-11-13,C0064,P005,North,5.0,4500,0.05,22500,1125.0,21375.0,14910.64,6464.36,30.242619883040934
O00175,2025-04-08,C0002,P001,North,1.0,85000,0.15,85000,12750.0,72250.0,51754.52,20495.48,28.367446366782005
O00180,2024-03-20,C0048,P009,South,5.0,1800,0.0,9000,0.0,9000.0,7109.29,1890.71,21.00788888888889


In [0]:
from pyspark.sql.functions import date_format
clean_df = clean_df.withColumn("Order_month",date_format("OrderDate","yyyy-MM"))





In [0]:
display(clean_df.select("OrderDate","Order_month","SalesAmount","Profit","profit_margin").limit(10))

OrderDate,Order_month,SalesAmount,Profit,profit_margin
2024-04-13,2024-04,113400.0,34978.31,30.84507054673721
2025-04-09,2025-04,4500.0,872.47,19.388222222222222
2024-10-26,2024-10,3200.0,867.16,27.09875
2025-06-29,2025-06,107800.0,45101.61,41.838228200371056
2025-11-14,2025-11,3528.0,1182.64,33.52154195011338
2025-06-24,2025-06,107800.0,22227.78,20.619461966604824
2024-11-13,2024-11,58800.0,22567.69,38.38042517006803
2025-11-13,2025-11,21375.0,6464.36,30.242619883040934
2025-04-08,2025-04,72250.0,20495.48,28.367446366782005
2024-03-20,2024-03,9000.0,1890.71,21.00788888888889


In [0]:
from pyspark.sql.functions import sum, round
region_summary=clean_df.groupBy("Region").agg(round(sum("SalesAmount"),2).alias("Total_Sales"),round(sum("Profit"),2).alias("Total_Profit"))



In [0]:
display(region_summary) 

Region,Total_Sales,Total_Profit
North,5.0216062E7,1.373727243E7
Unknown,686.0,273.32
East,2.2992907E7,6475853.7
South,3.1421068E7,8644040.3
West,2.7230804E7,7524702.61


In [0]:
region_summary=region_summary.withColumn("profit_margin",round((col("Total_Profit")/col("Total_Sales"))*100,2))
display(region_summary)


Region,Total_Sales,Total_Profit,profit_margin
North,5.0216062E7,1.373727243E7,27.36
Unknown,686.0,273.32,39.84
East,2.2992907E7,6475853.7,28.16
South,3.1421068E7,8644040.3,27.51
West,2.7230804E7,7524702.61,27.63


In [0]:
from pyspark.sql.functions import sum, round, col

product_summary = clean_df.groupBy("productid").agg(
    round(sum("salesamount"), 2).alias("total_sales"),
    round(sum("profit"), 2).alias("total_profit"),
    round(sum("quantity"), 2).alias("total_quantity")
)

product_summary = product_summary.withColumn(
    "profit_margin",
    round(
        (col("total_profit") / col("total_sales")) * 100,
        2
    )
)

display(product_summary)

productid,total_sales,total_profit,total_quantity,profit_margin
P004,1.13586E7,3063654.98,544.0,26.97
P011,955350.0,257554.56,457.0,26.96
P014,329903.0,99321.4,494.0,30.11
P009,934614.0,271681.84,543.0,29.07
P006,1285216.0,373023.85,421.0,29.02
P015,203185.0,58162.44,425.0,28.63
P010,562404.0,150269.16,492.0,26.72
P003,2.095632E7,6059320.89,523.0,28.91
P005,2567205.0,714694.98,596.0,27.84
P013,4798290.0,1326186.85,474.0,27.64


In [0]:
top_products = product_summary.orderBy(
    col("total_sales").desc()
).limit(10)

display(top_products)

productid,total_sales,total_profit,total_quantity,profit_margin
P001,4.11655E7,1.088805752E7,509.0,26.45
P002,3.021785E7,8365571.24,491.0,27.68
P003,2.095632E7,6059320.89,523.0,28.91
P004,1.13586E7,3063654.98,544.0,26.97
P012,6618090.0,1970782.8,477.0,29.78
P008,6152850.0,1715025.11,543.0,27.87
P013,4798290.0,1326186.85,474.0,27.64
P007,3756150.0,1068834.74,436.0,28.46
P005,2567205.0,714694.98,596.0,27.84
P006,1285216.0,373023.85,421.0,29.02


In [0]:
monthly_summary = clean_df.groupBy("order_month").agg(
    round(sum("salesamount"), 2).alias("total_sales"),
    round(sum("profit"), 2).alias("total_profit"),
    round(sum("quantity"), 2).alias("total_quantity")
)

monthly_summary = monthly_summary.withColumn(
    "profit_margin",
    round(
        (col("total_profit") / col("total_sales")) * 100,
        2
    )
)

display(monthly_summary.orderBy("order_month"))

order_month,total_sales,total_profit,total_quantity,profit_margin
2024-01,5705208.0,1607815.07,333.0,28.18
2024-02,4159686.0,1180367.07,249.0,28.38
2024-03,6618632.0,1953158.83,351.0,29.51
2024-04,7136861.0,2055831.56,338.0,28.81
2024-05,5568906.0,1539923.9,285.0,27.65
2024-06,5247806.0,1255284.77,298.0,23.92
2024-07,4623138.0,1270558.42,311.0,27.48
2024-08,6709694.0,1890922.02,359.0,28.18
2024-09,5636348.0,1459743.57,284.0,25.9
2024-10,5163741.0,1493388.1,304.0,28.92


In [0]:
customer_summary = clean_df.groupBy("customerId").agg(
    round(sum("salesamount"), 2).alias("total_sales"),
    round(sum("profit"), 2).alias("total_profit"),
    round(sum("quantity"), 2).alias("total_quantity")
)

display(customer_summary.orderBy(
    col("total_sales").desc()
).limit(10))

customerId,total_sales,total_profit,total_quantity
C0073,3086454.0,861579.69,95.0
C0081,2303232.0,682584.08,76.0
C0093,2282904.0,587150.89,122.0
C0025,2228603.0,720305.07,100.0
C0006,2220538.0,503570.79,101.0
C0055,2062010.0,468805.67,77.0
C0090,2022812.0,568634.58,61.0
C0023,1988258.0,503006.4,93.0
C0038,1978553.0,622342.36,79.0
C0100,1957292.0,564741.8,83.0


In [0]:
overall_summary = clean_df.agg(
    round(sum("salesamount"), 2).alias("total_sales"),
    round(sum("profit"), 2).alias("total_profit"),
    round(sum("quantity"), 2).alias("total_quantity")
)

display(overall_summary)

total_sales,total_profit,total_quantity
1.31861527E8,3.638214236E7,7425.0


In [0]:
overall_summary = overall_summary.withColumn(
    "profit_margin",
    round(
        (col("total_profit") / col("total_sales")) * 100,
        2
    )
)

display(overall_summary)

total_sales,total_profit,total_quantity,profit_margin
1.31861527E8,3.638214236E7,7425.0,27.59


In [0]:
print("Total rows:", clean_df.count())
print("Duplicate rows:",
      clean_df.count() - clean_df.distinct().count())

Total rows: 2500
Duplicate rows: 0


In [0]:
from pyspark.sql.functions import col, coalesce, lit

clean_df = clean_df.withColumn(
    "Quantity",
    coalesce(col("Quantity"), lit(0.0))
)

In [0]:
print(
    "Quantity NULLs:",
    clean_df.filter(col("Quantity").isNull()).count()
)

Quantity NULLs: 0


In [0]:
display(
    clean_df.filter(col("Quantity") == 0.0)
)

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit,profit_margin,Order_month
O00021,2024-08-22,C0055,P005,North,0.0,4500,0.15,13500,2025.0,11475.0,9261.07,2213.93,19.29350762527233,2024-08


In [0]:
display(
    clean_df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in clean_df.columns
    ])
)

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit,profit_margin,Order_month
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
clean_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/curated/retail_sales/")

In [0]:
curated_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("abfss://retail-data@retailsalesdata2026xxx.dfs.core.windows.net/curated/retail_sales/")

In [0]:
display(curated_df)

OrderID,OrderDate,CustomerID,ProductID,Region,Quantity,UnitPrice,Discount,GrossSales,DiscountAmount,SalesAmount,Cost,Profit,profit_margin,Order_month
O00029,2024-04-13,C0063,P003,North,3.0,42000,0.1,126000,12600.0,113400.0,78421.69,34978.31,30.84507054673721,2024-04-01
O00041,2025-04-09,C0044,P005,North,1.0,4500,0.0,4500,0.0,4500.0,3627.53,872.47,19.388222222222222,2025-04-01
O00073,2024-10-26,C0082,P006,South,1.0,3200,0.0,3200,0.0,3200.0,2332.84,867.16,27.09875,2024-10-01
O00078,2025-06-29,C0094,P004,North,5.0,22000,0.02,110000,2200.0,107800.0,62698.39,45101.61,41.838228200371056,2025-06-01
O00109,2025-11-14,C0013,P009,North,2.0,1800,0.02,3600,72.0,3528.0,2345.36,1182.64,33.52154195011338,2025-11-01
O00132,2025-06-24,C0068,P004,South,5.0,22000,0.02,110000,2200.0,107800.0,85572.22,22227.78,20.619461966604824,2025-06-01
O00135,2024-11-13,C0081,P008,North,5.0,12000,0.02,60000,1200.0,58800.0,36232.31,22567.69,38.38042517006803,2024-11-01
O00170,2025-11-13,C0064,P005,North,5.0,4500,0.05,22500,1125.0,21375.0,14910.64,6464.36,30.242619883040934,2025-11-01
O00175,2025-04-08,C0002,P001,North,1.0,85000,0.15,85000,12750.0,72250.0,51754.52,20495.48,28.367446366782005,2025-04-01
O00180,2024-03-20,C0048,P009,South,5.0,1800,0.0,9000,0.0,9000.0,7109.29,1890.71,21.00788888888889,2024-03-01
